In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = {}

df["train_orig"] = pd.read_csv("train.csv")
df["test_orig"]  = pd.read_csv("test.csv")

df["total_orig"] = pd.concat(objs=[df["train_orig"], df["test_orig"]], ignore_index=True)

In [3]:
df["total_orig"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(3), int64(4), object(5)
memory usage: 122.8+ KB


In [4]:
def create_total():
    df = globals()["df"]
    incomplete_columns = ["Cabin"]
    df["total"] = df["total_orig"].drop(labels=incomplete_columns, axis=1)
    
    unhelpful_columns = ["PassengerId", "Name", "Ticket"]

    df["total"] = df["total"].drop(labels=unhelpful_columns, axis=1)
    
    # drop_na_columns = ["Embarked"]
    drop_na_columns = []

    for column in drop_na_columns:
        df["total"] = df["total"][~df["total"][column].isna()]
        
    df["total"] = pd.get_dummies(df["total"], columns=["Sex", "Embarked"])

In [5]:
from sklearn.model_selection import train_test_split

def run_clf(clf, column = "Survived", drop_na_columns=False):
    df_total = globals()["df"]["total"]
    columns_with_na = df_total.columns[df_total.isna().any()]
    train    = df_total[~df_total[column].isna()]
    test     = df_total[ df_total[column].isna()].drop(column, axis=1)

    if drop_na_columns:        
        columns_with_na = test.columns[test.isna().any()]
        total_train = train.drop(columns_with_na, axis=1)
        total_test  = test.drop(columns_with_na, axis=1)
    else:
        total_train = train.dropna()
        total_test  = test.dropna()

    total_X_train = train.drop(column, axis=1)
    total_y_train = train[column]
    total_X_test  = test.copy()
    
    if total_X_test.shape[0] == 0: return []

    computed_column_name = column + "_computed"
    
    X_train, X_test, y_train, y_test = train_test_split(total_X_train, total_y_train, test_size=0.33, random_state=42)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc_log = clf.score(X_test, y_test) * 100
    print(f"{computed_column_name:20s} test set accuracy: {acc_log:.2f}%")

    total_y_pred = clf.predict(total_X_test)
    df_total[computed_column_name] = df_total[column]
    df_total.loc[total_X_test.index, computed_column_name] = total_y_pred
    df_total.drop(column, axis=1, inplace=True)

    return total_y_pred

In [6]:
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor

def impute_clf():
    return xgb.XGBRegressor()
#     return GradientBoostingRegressor()

def impute_missing():
    run_clf(impute_clf(), "Age")
    run_clf(impute_clf(), "Age_computed", drop_na_columns=True)
    
    run_clf(impute_clf(), "Fare")
    run_clf(impute_clf(), "Fare_computed", drop_na_columns=True)
    
    # Create attributes
#     df_total = globals()["df"]["total"]
#     df_total["Fare_over_Pclass"]  = df_total["Fare_computed"] / df_total["Pclass"]    

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB


random_state = 141

estimators = {
    "Random Forest":       RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=random_state),
    "K Nearest Neighbors": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(random_state=random_state),
    "Linear SVC":          LinearSVC(random_state=random_state),
    "Decision Tree":       DecisionTreeClassifier(random_state=random_state, max_depth=5),
    "Naive-Bayes":         GaussianNB(),
    "XGBoost Classifier":  xgb.XGBClassifier(n_estimators=1000, max_depth=5, random_state=random_state),
    
}

for name, classifier in estimators.items():
    print(f"======== {name} ========\n")
    create_total()
    impute_missing()
    result = run_clf(classifier, "Survived")
    print("========================\n\n")

# run_clf(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1), "Survived")
# run_clf(KNeighborsClassifier(), "Survived")

======== Random Forest ========

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 82.71%


======== K Nearest Neighbors ========

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 74.58%


======== Logistic Regression ========

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 81.69%


======== Linear SVC ========



/Users/peter.maneykowski/.pyenv/versions/3.9.13/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 75.59%


======== Decision Tree ========



/Users/peter.maneykowski/.pyenv/versions/3.9.13/lib/python3.9/site-packages/sklearn/svm/_base.py:1225: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 81.36%


======== Naive-Bayes ========

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 79.66%


======== XGBoost Classifier ========

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 80.34%




In [8]:
clf = RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=random_state)
# clf = xgb.XGBClassifier(max_depth=5, random_state=random_state)

create_total()
impute_missing()
run_clf(clf, "Survived")

Age_computed         test set accuracy: 10.75%
Fare_computed        test set accuracy: 40.73%
Survived_computed    test set accuracy: 82.71%


array([0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0.,
       0., 0., 1., 0., 0., 1., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0.,
       0., 1., 0., 1., 1., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0.,
       1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       1., 0., 0., 1., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 1.,
       0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 1., 0., 1., 1.

In [9]:
df["total"]["Survived_computed"] = df["total"]["Survived_computed"].round().astype(int)
df["total"]["PassengerId"] = df["total_orig"]["PassengerId"]
test_orig = df["total_orig"][df["total_orig"]["Survived"].isna()]
df["total"]["Survived"] = df["total"]["Survived_computed"]
df["total"].loc[test_orig.index][["PassengerId", "Survived_computed"]].to_csv(
    "submission.csv",
    header=["PassengerId", "Survived"],
    index=False
)

In [10]:
def survived_for_attr(df, column):
    return df[[column, 'Survived']].groupby(
        [column],
        as_index=False
    ).mean().sort_values(
        by='Survived',
        ascending=False
    )

In [11]:
df["total_orig"]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1305,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
1306,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [12]:
survived_for_attr(df["total_orig"], "Sex")

,Sex,Survived
0,female,0.742038
1,male,0.188908
